# 01b. 자동 라벨링 (Grounding DINO) → Roboflow 검수용 zip

`01_collect_and_clean.ipynb`에서 만든 `dataset_clean.zip`에 **초기 박스를 자동으로 그려서** Roboflow에서 검수·수정만 하면 되게 만듭니다.

**방식**
- 글자로 대상을 지정하는 zero-shot 탐지 모델 **Grounding DINO**로 `"bee. wasp. hornet."`을 찾아 박스를 만듭니다.
- 이 모델은 장수말벌과 등검은말벌을 구분하지 못하므로, **클래스는 폴더 이름으로 정합니다** (폴더가 이미 부류별로 나뉘어 있음).
- 말벌 폴더 사진에 꿀벌이 같이 찍힌 경우: `bee`로 탐지되고, 그 사진에서 가장 큰 박스보다 확실히 작은 박스는 `honeybee`로 둡니다.
- `hard_negative`·`background`는 박스 없는(Null) 이미지로 넣습니다.
- 박스를 하나도 못 찾은 부류 사진은 따로 모아 **손으로 라벨링**합니다.

**결과물**
- `autolabel_roboflow.zip` — YOLOv8 형식 (이미지 + 라벨 + data.yaml). Roboflow **Upload Data**에 그대로 끌어다 놓으면 박스가 붙은 채로 올라갑니다.
- `needs_manual.zip` — 박스를 못 찾은 사진. Roboflow에 따로 올려 직접 박스를 칩니다.
- `review_list.csv` — 꿀벌·말벌이 섞였거나 박스가 많은 사진 목록 (검수 때 우선 확인)

**검수는 필수입니다.** 자동 박스는 펼친 날개까지 넣거나, 둥지·다른 곤충에 박스를 치기도 합니다. Roboflow에서 모든 사진의 박스 위치와 클래스를 확인하고 고친 뒤 데이터셋에 넣으세요.
보고서에는 "Grounding DINO로 초기 박스를 생성하고 전수 검수해 위치와 클래스를 수정함"처럼 방법을 적습니다.

**런타임 → 런타임 유형 변경 → T4 GPU**에서 실행하세요.

In [ ]:
# ===== 셀 1: 환경 준비 =====
!pip -q install -U transformers
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "(cpu면 매우 느립니다. 런타임 유형을 T4 GPU로 바꾸세요)")

In [ ]:
# ===== 셀 2: dataset_clean.zip 가져오기 =====
# 01 노트북 셀 7에서 받은 zip을 업로드합니다. Drive에 저장했다면 DRIVE_ZIP에 경로를 적으세요.
import os, zipfile, shutil
DRIVE_ZIP = ""   # 예: "/content/drive/MyDrive/beeguard/dataset_clean.zip"
CLEAN = "/content/clean"

if DRIVE_ZIP:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = DRIVE_ZIP
else:
    from google.colab import files
    up = files.upload()
    zip_path = next(iter(up))
shutil.rmtree(CLEAN, ignore_errors=True)
zipfile.ZipFile(zip_path).extractall(CLEAN)
for g in sorted(os.listdir(CLEAN)):
    if os.path.isdir(os.path.join(CLEAN, g)):
        print(f"{g:<22}{len(os.listdir(os.path.join(CLEAN, g))):>5}장")

In [ ]:
# ===== 셀 3: 설정 =====
MODEL_ID = "IDEA-Research/grounding-dino-base"
TEXT = "bee. wasp. hornet."      # 찾을 대상 (소문자, 마침표로 구분)
BOX_THRESHOLD = 0.30             # 낮추면 더 많이 찾지만 엉뚱한 박스도 늘어남
TEXT_THRESHOLD = 0.25
DEDUP_IOU = 0.5                  # 이보다 많이 겹치는 박스는 같은 개체로 보고 하나만 남김
MAX_BOX_FRAC = 0.9               # 사진 넓이의 90% 이상을 덮는 박스는 버림
BEE_SMALL_RATIO = 0.5            # 말벌 폴더에서 'bee' 박스가 가장 큰 박스 넓이의 이 비율 미만이면 honeybee로

NAMES = ["giant_hornet", "honeybee", "yellow_legged_hornet"]   # Roboflow 알파벳순과 같은 순서
CLASS_GROUPS = {g: i for i, g in enumerate(NAMES)}             # 폴더 이름 → 클래스 번호
NULL_GROUPS = ["hard_negative", "background"]                  # 박스 없이 넣을 폴더

In [ ]:
# ===== 셀 4: 자동 박스 생성 =====
import glob, time
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

proc = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(DEVICE).eval()

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def detect(path):
    im = Image.open(path).convert("RGB")
    W, H = im.size
    inp = proc(images=im, text=TEXT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inp)
    r = proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=BOX_THRESHOLD,
                                                    text_threshold=TEXT_THRESHOLD, target_sizes=[(H, W)])[0]
    words = r.get("text_labels", r.get("labels"))
    dets = []
    for b, s, w in sorted(zip(r["boxes"].tolist(), r["scores"].tolist(), words), key=lambda x: -x[1]):
        x0, y0, x1, y1 = max(0, b[0]), max(0, b[1]), min(W, b[2]), min(H, b[3])
        if x1 <= x0 or y1 <= y0 or (x1 - x0) * (y1 - y0) > MAX_BOX_FRAC * W * H:
            continue
        if all(iou((x0, y0, x1, y1), d["box"]) < DEDUP_IOU for d in dets):
            dets.append({"box": (x0, y0, x1, y1), "score": s, "word": str(w)})
    return dets, W, H

def assign(group, dets):
    """폴더 이름으로 클래스를 정하고, 말벌 폴더의 작은 'bee' 박스만 honeybee로 바꾼다."""
    if not dets:
        return []
    big = max((d["box"][2] - d["box"][0]) * (d["box"][3] - d["box"][1]) for d in dets)
    out = []
    for d in dets:
        cls = CLASS_GROUPS[group]
        area = (d["box"][2] - d["box"][0]) * (d["box"][3] - d["box"][1])
        if group != "honeybee" and "bee" in d["word"] and area < BEE_SMALL_RATIO * big:
            cls = CLASS_GROUPS["honeybee"]
        out.append((cls, d["box"], d["score"]))
    return out

LABELS, SIZES = {}, {}
t0 = time.time()
for group in CLASS_GROUPS:
    files_ = sorted(glob.glob(os.path.join(CLEAN, group, "*.jpg")))
    for k, f in enumerate(files_, 1):
        dets, W, H = detect(f)
        LABELS[f], SIZES[f] = assign(group, dets), (W, H)
        if k % 100 == 0:
            print(f"  {group}: {k}/{len(files_)}  ({time.time() - t0:.0f}초)")
    nb = [len(LABELS[f]) for f in files_]
    print(f"[{group}] {len(files_)}장 | 박스 {sum(nb)}개 | 못 찾음 {nb.count(0)}장 | 2개 이상 {sum(n > 1 for n in nb)}장")
print(f"완료: {time.time() - t0:.0f}초")

In [ ]:
# ===== 셀 5: 미리보기 (부류마다 무작위 16장) =====
import random
from PIL import ImageDraw
from IPython.display import display
COLORS = {0: (230, 60, 40), 1: (250, 190, 0), 2: (120, 60, 200)}   # giant_hornet / honeybee / yellow_legged_hornet

def preview(group, n=16, cols=8, t=220):
    fs = [f for f in LABELS if os.path.basename(os.path.dirname(f)) == group]
    fs = random.Random(0).sample(fs, min(n, len(fs)))
    sheet = Image.new("RGB", (cols * t, ((len(fs) + cols - 1) // cols) * t), "white")
    for k, f in enumerate(fs):
        im = Image.open(f).convert("RGB"); d = ImageDraw.Draw(im)
        for cls, b, s in LABELS[f]:
            d.rectangle(b, outline=COLORS[cls], width=max(3, im.size[0] // 120))
        im.thumbnail((t, t)); sheet.paste(im, ((k % cols) * t, (k // cols) * t))
    print(group); display(sheet)

for g in CLASS_GROUPS:
    preview(g)
print("색: 빨강 = giant_hornet, 노랑 = honeybee, 보라 = yellow_legged_hornet")

In [ ]:
# ===== 셀 6: Roboflow 업로드용 zip 만들기 =====
import csv, yaml
OUT, MANUAL = "/content/autolabel", "/content/needs_manual"
shutil.rmtree(OUT, ignore_errors=True); shutil.rmtree(MANUAL, ignore_errors=True)
os.makedirs(f"{OUT}/train/images"); os.makedirs(f"{OUT}/train/labels"); os.makedirs(MANUAL)

def put(src, lines):
    name = os.path.basename(src)
    shutil.copy(src, f"{OUT}/train/images/{name}")
    with open(f"{OUT}/train/labels/{os.path.splitext(name)[0]}.txt", "w") as fp:
        fp.write("\n".join(lines) + ("\n" if lines else ""))

n_auto = n_null = n_manual = 0
review = []
for f, labs in LABELS.items():
    if not labs:                                     # 못 찾은 사진 → 손으로 라벨링
        shutil.copy(f, MANUAL); n_manual += 1; continue
    W, H = SIZES[f]
    lines = [f"{c} {(b[0] + b[2]) / 2 / W:.6f} {(b[1] + b[3]) / 2 / H:.6f} {(b[2] - b[0]) / W:.6f} {(b[3] - b[1]) / H:.6f}" for c, b, s in labs]
    put(f, lines); n_auto += 1
    classes = {c for c, _, _ in labs}
    if len(classes) > 1 or len(labs) >= 4:
        review.append([os.path.basename(f), len(labs), " ".join(NAMES[c] for c in sorted(classes))])
for g in NULL_GROUPS:                                # 박스 없는 사진: 빈 라벨 파일
    for f in sorted(glob.glob(os.path.join(CLEAN, g, "*.jpg"))):
        put(f, []); n_null += 1

yaml.safe_dump({"train": "train/images", "val": "train/images", "nc": len(NAMES), "names": NAMES},
               open(f"{OUT}/data.yaml", "w"), sort_keys=False)
with open("/content/review_list.csv", "w", newline="", encoding="utf-8-sig") as fp:
    wr = csv.writer(fp); wr.writerow(["file", "boxes", "classes"]); wr.writerows(review)
shutil.make_archive("/content/autolabel_roboflow", "zip", OUT)
if n_manual:
    shutil.make_archive("/content/needs_manual", "zip", MANUAL)
print(f"자동 박스 {n_auto}장 + Null {n_null}장 → autolabel_roboflow.zip")
print(f"손으로 라벨링할 사진 {n_manual}장 → needs_manual.zip")
print(f"우선 검수할 사진 {len(review)}장 → review_list.csv")
try:
    from google.colab import files
    files.download("/content/autolabel_roboflow.zip")
    if n_manual:
        files.download("/content/needs_manual.zip")
    files.download("/content/review_list.csv")
except ImportError:
    pass

## AI 검수 (선택)
셀 7이 박스마다 번호를 붙인 시트(`review_sheets.zip`)를 만듭니다. 이 zip을 Claude에게 올리면 시트를 검수해 `corrections.json`(박스 삭제·클래스 수정·사진 제외)을 돌려줍니다.
셀 8에 `corrections.json`을 올리면 `autolabel_reviewed.zip`이 만들어지고, 이걸 Roboflow에 올립니다.

AI 검수는 박스 크기 미세 조정(펼친 날개 제외 등)이나 놓친 개체 추가는 하지 못합니다. 보고서에는 검수 방법을 그대로 적으세요.

In [ ]:
# ===== 셀 7: AI 검수용 시트 만들기 =====
# 박스마다 번호(#0, #1 …)를 붙인 그림을 12장씩 모아 review_sheets.zip으로 내려받습니다.
# 이 zip을 Claude에게 올리면 검수해서 corrections.json(박스 삭제·클래스 수정·사진 제외)을 만들어 줍니다.
# 셀 6을 실행한 뒤라면 바로, 런타임이 끊겼다면 autolabel_roboflow.zip을 /content/autolabel에 다시 풀고 실행하세요.
import os, glob, shutil, zipfile
from PIL import Image, ImageDraw, ImageFont
OUT = "/content/autolabel"
if not os.path.isdir(f"{OUT}/train/labels"):
    from google.colab import files
    print("autolabel_roboflow.zip을 올려 주세요"); up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall(OUT)
NAMES = ["giant_hornet", "honeybee", "yellow_legged_hornet"]
COLORS = {0: (230, 40, 30), 1: (255, 200, 0), 2: (160, 60, 230)}
SHORT = {0: "GIANT", 1: "BEE", 2: "YELLOW"}
def font(size):
    for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "DejaVuSans-Bold.ttf"]:
        try: return ImageFont.truetype(p, size)
        except OSError: pass
    return ImageFont.load_default()
F_BOX, F_CAP = font(26), font(18)

items = []
for lp in sorted(glob.glob(f"{OUT}/train/labels/*.txt")):
    stem = os.path.splitext(os.path.basename(lp))[0]
    lines = [l.split() for l in open(lp) if l.strip()]
    if lines:                                         # Null(박스 없음) 사진은 제외
        items.append((stem, lines))
T, COLS, ROWS = 400, 4, 3
SHEETS = "/content/review_sheets"; shutil.rmtree(SHEETS, ignore_errors=True); os.makedirs(SHEETS)
for s in range(0, len(items), COLS * ROWS):
    sheet = Image.new("RGB", (COLS * T, ROWS * (T + 26)), "white"); ds = ImageDraw.Draw(sheet)
    for k, (stem, lines) in enumerate(items[s:s + COLS * ROWS]):
        im = Image.open(glob.glob(f"{OUT}/train/images/{stem}.*")[0]).convert("RGB")
        W, H = im.size; sc = T / max(W, H); im = im.resize((max(1, int(W * sc)), max(1, int(H * sc))))
        d = ImageDraw.Draw(im)
        for i, p in enumerate(lines):
            c, cx, cy, w, h = int(p[0]), *map(float, p[1:5])
            x0, y0, x1, y1 = (cx - w / 2) * im.width, (cy - h / 2) * im.height, (cx + w / 2) * im.width, (cy + h / 2) * im.height
            d.rectangle((x0, y0, x1, y1), outline=COLORS[c], width=3)
            tag = f"#{i} {SHORT[c]}"; tw = d.textlength(tag, font=F_BOX)
            ty = y0 - 28 if y0 > 28 else y0 + 2
            d.rectangle((x0, ty, x0 + tw + 6, ty + 28), fill=COLORS[c]); d.text((x0 + 3, ty), tag, fill="black", font=F_BOX)
        x, y = (k % COLS) * T, (k // COLS) * (T + 26)
        sheet.paste(im, (x, y)); ds.text((x + 4, y + T + 3), stem, fill="black", font=F_CAP)
    sheet.save(f"{SHEETS}/sheet_{s // (COLS * ROWS) + 1:03d}.jpg", quality=82)
shutil.make_archive("/content/review_sheets", "zip", SHEETS)
print(f"사진 {len(items)}장 → 시트 {len(os.listdir(SHEETS))}장, review_sheets.zip {os.path.getsize('/content/review_sheets.zip') // 2**20} MB")
try:
    from google.colab import files
    files.download("/content/review_sheets.zip")
except ImportError:
    pass

In [ ]:
# ===== 셀 8: corrections.json 적용 → 검수된 zip 만들기 =====
# corrections.json 형식: {"파일이름(확장자 없이)": {"drop": true} 또는 {"delete": [박스 번호], "relabel": {"박스 번호": "클래스 이름"}}}
import json
from google.colab import files
up = files.upload()                                   # corrections.json 선택
corr = json.load(open(next(iter(up)), encoding="utf-8"))
REV = "/content/autolabel_reviewed"
shutil.rmtree(REV, ignore_errors=True); shutil.copytree(OUT, REV)
stats = dict(dropped=0, box_deleted=0, relabeled=0, missing=0)
for stem, c in corr.items():
    lp = f"{REV}/train/labels/{stem}.txt"
    if not os.path.exists(lp):
        stats["missing"] += 1; continue
    if c.get("drop"):
        os.remove(lp)
        for ip in glob.glob(f"{REV}/train/images/{stem}.*"): os.remove(ip)
        stats["dropped"] += 1; continue
    lines = [l.split() for l in open(lp) if l.strip()]
    for i, cls in c.get("relabel", {}).items():
        if int(i) < len(lines):
            lines[int(i)][0] = str(NAMES.index(cls)); stats["relabeled"] += 1
    dele = {int(i) for i in c.get("delete", []) if int(i) < len(lines)}
    stats["box_deleted"] += len(dele)
    lines = [l for i, l in enumerate(lines) if i not in dele]
    if not lines:                                     # 박스를 다 지웠으면 대상이 없는 사진 → 제외
        os.remove(lp)
        for ip in glob.glob(f"{REV}/train/images/{stem}.*"): os.remove(ip)
        stats["dropped"] += 1; continue
    open(lp, "w").write("\n".join(" ".join(l) for l in lines) + "\n")
n_img = len(glob.glob(f"{REV}/train/images/*"))
shutil.make_archive("/content/autolabel_reviewed", "zip", REV)
print(stats, f"→ 남은 사진 {n_img}장, autolabel_reviewed.zip 생성")
files.download("/content/autolabel_reviewed.zip")